# Stage 3: DPO Preference Alignment
**Goal:** Improve response quality using preference data (chosen vs rejected).

## Step 1 — Install dependencies

In [1]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## Step 2 — Imports

In [2]:
import torch
import json
from unsloth import FastLanguageModel, PatchDPOTrainer
from datasets import Dataset
from trl import DPOTrainer, DPOConfig
from transformers import TrainingArguments

PatchDPOTrainer()   # Unsloth patch for DPO compatibility
print(f"GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4


## Step 3 — Configuration

In [8]:
SFT_ADAPTER_PATH = "/content/drive/MyDrive/finance_ft/stage2_sft_adapter"  # Load from Stage 2
MAX_SEQ_LEN      = 512
LOAD_IN_4BIT     = True
LORA_R           = 16
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.05
TARGET_MODS      = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "up_proj", "down_proj", "gate_proj"]
OUTPUT_DIR       = "./outputs/stage3_dpo"
EPOCHS           = 1       # DPO needs fewer epochs than SFT
BATCH_SIZE       = 2
GRAD_ACCUM       = 4
LR               = 5e-5   # Lower LR for DPO alignment
BETA             = 0.1    # DPO temperature — controls deviation from reference model

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 4 — Load SFT model

In [10]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = SFT_ADAPTER_PATH,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = TARGET_MODS,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)
print("SFT model loaded for DPO training")

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.7.2 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


SFT model loaded for DPO training


## Step 5 — Load preference dataset

In [12]:
from datasets import load_dataset
from datasets import Dataset

ds = load_dataset("gbharti/finance-alpaca")

all_records = [
    item for item in ds["train"]
    if len(item["instruction"].strip()) > 10
    and len(item["output"].strip()) > 50
]

# 50 unseen preference pairs
preference_records = all_records[200:250]

dataset = Dataset.from_dict({
    "prompt"  : [r["instruction"].strip() for r in preference_records],
    "chosen"  : [r["output"].strip() for r in preference_records],
    "rejected": [
        r["output"].strip().split(".")[0] + ". This depends on various factors."
        for r in preference_records
    ],
})

print(f"Preference pairs: {len(dataset)}")
print(f"\nSample prompt  : {dataset['prompt'][0]}")
print(f"Sample chosen  : {dataset['chosen'][0][:200]}")
print(f"Sample rejected: {dataset['rejected'][0]}")

Preference pairs: 50

Sample prompt  : How to learn about doing technical analysis?  Any suggested programs or tools that teach it?
Sample chosen  : I recall the name Martin Pring. As my fundamental analysis book from grad school was the work of Graham and Dodd titled Security Analysis, Pring was the author of the books I read on technical analysi
Sample rejected: I recall the name Martin Pring. This depends on various factors.


## Step 6 — Train with DPO

In [13]:
dpo_config = DPOConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    beta                        = BETA,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 5,
    save_strategy               = "epoch",
    warmup_ratio                = 0.1,
    report_to                   = "none",
    max_length                  = MAX_SEQ_LEN,
    max_prompt_length           = 256,
)

trainer = DPOTrainer(
    model     = model,
    args      = dpo_config,
    train_dataset = dataset,
    tokenizer = tokenizer,
)

print("Starting DPO alignment...")
trainer_stats = trainer.train()
print(f"DPO complete. Final loss: {trainer_stats.training_loss:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Starting DPO alignment...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 1 | Total steps = 7
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,9.887289,-12.637381,-3.055595,0.125000,-9.581786,-753.098999,-130.095047,-2.775886,-2.847138


Unsloth: Restored added_tokens_decoder metadata in ./outputs/stage3_dpo/checkpoint-7/tokenizer_config.json.


DPO complete. Final loss: 9.2123


## Step 7 — Save DPO model

In [14]:
DPO_ADAPTER_PATH = "./outputs/stage3_dpo_adapter"
model.save_pretrained(DPO_ADAPTER_PATH)
tokenizer.save_pretrained(DPO_ADAPTER_PATH)
print(f"DPO adapter saved to: {DPO_ADAPTER_PATH}")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    model.save_pretrained("/content/drive/MyDrive/finance_ft/stage3_dpo_adapter")
    print("DPO adapter also saved to Google Drive")
except:
    print("Not on Colab — skipping Drive save")

Unsloth: Restored added_tokens_decoder metadata in ./outputs/stage3_dpo_adapter/tokenizer_config.json.


DPO adapter saved to: ./outputs/stage3_dpo_adapter
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DPO adapter also saved to Google Drive


## Step 8 — Final inference test (10 questions)

In [ ]:
ALPACA_TEMPLATE_INFERENCE = """Below is a finance question. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
"""

FastLanguageModel.for_inference(model)

test_questions = [
    "What is a mutual fund SIP?",
    "What is the difference between a savings account and a fixed deposit?",
    "How does compound interest work?",
    "What is a credit score and why does it matter?",
    "How can I save tax under Section 80C?",
    "What is the repo rate and how does it affect loans?",
    "What is the difference between term insurance and whole life insurance?",
    "How does the stock market work?",
    "What is a demat account and how do I open one?",
    "What is the difference between a mutual fund and an ETF?",
]

dpo_outputs = {}
for q in test_questions:
    prompt = ALPACA_TEMPLATE_INFERENCE.format(q)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 150,
        temperature    = 0.3,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = answer.split("### Answer:")[-1].strip()
    dpo_outputs[q] = answer
    print(f"Q: {q}")
    print(f"A: {answer[:200]}")
    print("-" * 60)

In [15]:
from datasets import load_dataset

ds = load_dataset("gbharti/finance-alpaca")

# Filter valid records same way as training
all_records = [
    item for item in ds["train"]
    if len(item["instruction"].strip()) > 10
    and len(item["output"].strip()) > 50
]

# Training used first 100 → test from 500 onwards (safe gap)
test_records = all_records[500:510]  # 10 unseen examples

# Extract just the questions
test_questions = [r["instruction"].strip() for r in test_records]
test_answers   = [r["output"].strip() for r in test_records]  # ground truth

print("Test questions (unseen during training):")
for i, q in enumerate(test_questions):
    print(f"\n{i+1}. {q}")

Test questions (unseen during training):

1. Does wash sale apply if I buy stock on 2 two different dates and sell it later

2. Short Term Capital Gains tax vs. IRA Withdrawal Tax w/o Quarterly Est. Taxes

3. Are there any disadvantages to DHA Investment Properties?

4. Stock Trade Transaction Fee - at what point is it worth it

5. Why would a car company lend me money at a very low interest rate?

6. Swap hedging a currency hedge

7. Under what circumstance will the IRS charge you a late-payment penalty for taxes?

8. Buy car vs lease vs long term rent for 10 years period

9. How do dividend reinvestment purchases work?

10. Are traders 100% responsible for a stock's price changes?


In [16]:
from unsloth import FastLanguageModel

# Step 1 — Load DPO adapter from Drive
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "/content/drive/MyDrive/finance_ft/stage3_dpo_adapter",
    max_seq_length = 512,
    dtype          = None,
    load_in_4bit   = True,
)

# Step 2 — Switch to inference mode
FastLanguageModel.for_inference(model)

# Step 3 — Run on 10 unseen test questions
ALPACA_TEMPLATE_INFERENCE = """Below is a finance question. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
"""

dpo_outputs = {}
for q in test_questions:
    prompt  = ALPACA_TEMPLATE_INFERENCE.format(q)
    inputs  = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 150,
        temperature    = 0.3,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = answer.split("### Answer:")[-1].strip()
    dpo_outputs[q] = answer

    print(f"Q: {q}")
    print(f"A: {answer[:200]}")
    print("-" * 60)

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_util

Q: Does wash sale apply if I buy stock on 2 two different dates and sell it later
A: Wash sales are only applicable if you buy the same security at a later date and sell it again.  If you buy on day 1 and then sell on day 2, you have a wash sale.  If you bought on day 2 and then sold 
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: Under what circumstance will the IRS charge you a late-payment penalty for taxes?
A: The IRS will assess a late payment penalty for any unpaid tax due if it is not paid within 3 years from the due date. The due date is the 15th of the 4th month after the end of the tax year. The 3 yea
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Buy car vs lease vs long term rent for 10 years period
A: I think you should consider the cost of insurance as well. Insurance is expensive, and you should plan on paying at least 1x your annual income. If you are talking about a single car, I'd get a used F
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do dividend reinvestment purchases work?
A: You would place a limit order to buy shares at a specific price, then when the shares are traded you would place a second order to sell them at a different price. The second price is the price at whic
------------------------------------------------------------
Q: Are traders 100% responsible for a stock's price changes?
A: No. Traders are not 100% responsible for stock price changes.  Instead, stock price is determined by supply and demand.  When there is a demand for a stock, there is a demand for that stock.  When the
------------------------------------------------------------


In [18]:
!pip install bert_score -q
!pip install rouge_score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [19]:
# ── ROUGE ──
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

r1, r2, rL = [], [], []
for i, q in enumerate(test_questions):
    scores = scorer.score(test_answers[i], dpo_outputs[q])
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

dpo_rouge = {
    "rouge1": sum(r1)/len(r1),
    "rouge2": sum(r2)/len(r2),
    "rougeL": sum(rL)/len(rL),
}
print(f"DPO ROUGE-1: {dpo_rouge['rouge1']:.3f}")
print(f"DPO ROUGE-2: {dpo_rouge['rouge2']:.3f}")
print(f"DPO ROUGE-L: {dpo_rouge['rougeL']:.3f}")

# ── BERTScore ──
from bert_score import score as bert_score
import torch

P, R, F1 = bert_score(
    [dpo_outputs[q] for q in test_questions],
    test_answers,
    lang    = "en",
    device  = "cuda",
    verbose = False
)
dpo_bert = F1.mean().item()
print(f"DPO BERTScore F1: {dpo_bert:.3f}")

# ── Final comparison table ──
print("\n" + "="*65)
print("COMPLETE EVALUATION — ALL STAGES")
print("="*65)
print(f"{'Model':<28} {'ROUGE-1':>8} {'ROUGE-2':>8} {'ROUGE-L':>8} {'BERT-F1':>8}")
print("-"*65)
print(f"{'Base model':<28} {'0.180':>8} {'0.030':>8} {'0.136':>8} {'0.804':>8}")
print(f"{'Stage 1 non-instruction':<28} {'0.226':>8} {'0.033':>8} {'0.139':>8} {'0.819':>8}")
print(f"{'Stage 2 SFT':<28} {'0.224':>8} {'0.042':>8} {'0.133':>8} {'0.828':>8}")
print(f"{'Stage 3 DPO':<28} {dpo_rouge['rouge1']:>8.3f} {dpo_rouge['rouge2']:>8.3f} {dpo_rouge['rougeL']:>8.3f} {dpo_bert:>8.3f}")
print("="*65)

DPO ROUGE-1: 0.224
DPO ROUGE-2: 0.038
DPO ROUGE-L: 0.133


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DPO BERTScore F1: 0.828

COMPLETE EVALUATION — ALL STAGES
Model                         ROUGE-1  ROUGE-2  ROUGE-L  BERT-F1
-----------------------------------------------------------------
Base model                      0.180    0.030    0.136    0.804
Stage 1 non-instruction         0.226    0.033    0.139    0.819
Stage 2 SFT                     0.224    0.042    0.133    0.828
Stage 3 DPO                     0.224    0.038    0.133    0.828
